<a href="https://colab.research.google.com/github/huydinh210299/BT1NhanDangMau/blob/main/Nh%E1%BA%ADn_d%E1%BA%A1ng_m%E1%BA%ABu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

file_path = "/content/drive/MyDrive/Nhan_dang_mau/bonbanh_car_prices_2000.csv"

df = pd.read_csv(file_path)
df.head()

,listing_id,title,brand,condition,year,price_million_vnd,location,origin,color,fuel,engine_litre,transmission,seats,mileage_km,source_url
0,6976005,Lexus LM 300h Royal Lounge - 2020,Lexus,cũ,2020,4500,Cần Thơ,nhập khẩu,đen,hybrid,2.5,tự động,4.0,35000.0,https://bonbanh.com/oto/generic
1,6976002,VinFast VF2 Eco - 2026,Audi,mới,2026,169,Hà Nội,lắp ráp trong nước,xanh,điện,NaN,tự động,4.0,NaN,https://bonbanh.com/oto/audi
2,6975999,Honda Civic 1.8 AT - 2008,Nissan,cũ,2008,165,Đồng Nai,lắp ráp trong nước,xám,xăng,1.8,tự động,5.0,999999.0,https://bonbanh.com/oto/nissan
3,6975994,Honda CRV e:HEV RS - 2026,Volkswagen,cũ,2026,1210,Hà Nội,lắp ráp trong nước,đỏ,hybrid,2.0,tự động,5.0,11000.0,https://bonbanh.com/oto/volkswagen
4,6975985,Kia Cerato 2.0 AT - 2018,Kia,cũ,2018,365,Hà Nội,lắp ráp trong nước,cát,xăng,2.0,tự động,5.0,89000.0,https://bonbanh.com/oto/generic


In [3]:
print(df.shape)
display(df.info())
display(df.isna().sum().sort_values(ascending=False))
display(df.describe(include="all").T)

(2000, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   listing_id         2000 non-null   int64  
 1   title              2000 non-null   object 
 2   brand              2000 non-null   object 
 3   condition          2000 non-null   object 
 4   year               2000 non-null   int64  
 5   price_million_vnd  2000 non-null   int64  
 6   location           2000 non-null   object 
 7   origin             2000 non-null   object 
 8   color              1993 non-null   object 
 9   fuel               1999 non-null   object 
 10  engine_litre       1875 non-null   float64
 11  transmission       1999 non-null   object 
 12  seats              1999 non-null   float64
 13  mileage_km         1552 non-null   float64
 14  source_url         2000 non-null   object 
dtypes: float64(3), int64(3), object(9)
memory usage: 234.5+ KB


None

,0
mileage_km,448
engine_litre,125
color,7
fuel,1
seats,1
transmission,1
listing_id,0
location,0
price_million_vnd,0
year,0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
listing_id,2000.0,NaN,NaN,NaN,6959373.231,13005.470329,6934729.0,6948157.75,6959384.5,6972265.25,6976005.0
title,2000,1194,VinFast VF5 Plus - 2026,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
brand,2000,42,Toyota,380,NaN,NaN,NaN,NaN,NaN,NaN,NaN
condition,2000,2,cũ,1826,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,2000.0,NaN,NaN,NaN,2020.206,4.717488,1997.0,2018.0,2021.0,2024.0,2026.0
price_million_vnd,2000.0,NaN,NaN,NaN,975.486,1313.179636,28.0,399.0,612.5,1020.0,28000.0
location,2000,38,Hà Nội,934,NaN,NaN,NaN,NaN,NaN,NaN,NaN
origin,2000,2,lắp ráp trong nước,1129,NaN,NaN,NaN,NaN,NaN,NaN,NaN
color,1993,14,trắng,760,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fuel,1999,4,xăng,1479,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
target = "price_million_vnd"

features = [
    "brand", "condition", "year", "location", "origin", "color",
    "fuel", "engine_litre", "transmission", "seats", "mileage_km"
]

X = df[features]
y = df[target]

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = ["year", "engine_litre", "seats", "mileage_km"]
categorical_features = [
    "brand", "condition", "location", "origin", "color",
    "fuel", "transmission"
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [8]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate

cv = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Baseline (Median)": DummyRegressor(strategy="median"),
    "Ridge": Ridge(alpha=10),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, random_state=42, n_jobs=-1,
        min_samples_leaf=2
    )
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2"
        }
    )

    results.append({
        "Model": name,
        "MAE CV": -scores["test_mae"].mean(),
        "RMSE CV": -scores["test_rmse"].mean(),
        "RMSE Std": scores["test_rmse"].std(),
        "R2 CV": scores["test_r2"].mean()
    })

results_df = pd.DataFrame(results).sort_values("RMSE CV")
results_df

,Model,MAE CV,RMSE CV,RMSE Std,R2 CV
2,Random Forest,222.018009,845.608098,421.589697,0.582496
1,Ridge,485.125155,1049.723464,374.423016,0.383921
0,Baseline (Median),593.133125,1376.074250,340.397248,-0.093539


In [9]:
from sklearn.decomposition import PCA

ridge_pca = Pipeline([
    ("preprocess", preprocessor),
    ("pca", PCA(n_components=0.95, random_state=42)),
    ("model", Ridge(alpha=10))
])

scores_pca = cross_validate(
    ridge_pca, X_train, y_train, cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }
)

pca_result = pd.DataFrame([{
    "Model": "Ridge + PCA (95% variance)",
    "MAE CV": -scores_pca["test_mae"].mean(),
    "RMSE CV": -scores_pca["test_rmse"].mean(),
    "RMSE Std": scores_pca["test_rmse"].std(),
    "R2 CV": scores_pca["test_r2"].mean()
}])

comparison = pd.concat([results_df, pca_result]).sort_values("RMSE CV")
comparison

,Model,MAE CV,RMSE CV,RMSE Std,R2 CV
2,Random Forest,222.018009,845.608098,421.589697,0.582496
1,Ridge,485.125155,1049.723464,374.423016,0.383921
0,Ridge + PCA (95% variance),517.565986,1093.310471,355.465719,0.317107
0,Baseline (Median),593.133125,1376.074250,340.397248,-0.093539


In [11]:
from sklearn.decomposition import PCA

ridge_pca = Pipeline([
    ("preprocess", preprocessor),
    ("pca", PCA(n_components=0.95, random_state=42)),
    ("model", Ridge(alpha=10))
])

scores_pca = cross_validate(
    ridge_pca, X_train, y_train, cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }
)

pca_result = pd.DataFrame([{
    "Model": "Ridge + PCA (95% variance)",
    "MAE CV": -scores_pca["test_mae"].mean(),
    "RMSE CV": -scores_pca["test_rmse"].mean(),
    "RMSE Std": scores_pca["test_rmse"].std(),
    "R2 CV": scores_pca["test_r2"].mean()
}])

comparison = pd.concat([results_df, pca_result]).sort_values("RMSE CV")
comparison

,Model,MAE CV,RMSE CV,RMSE Std,R2 CV
2,Random Forest,222.018009,845.608098,421.589697,0.582496
1,Ridge,485.125155,1049.723464,374.423016,0.383921
0,Ridge + PCA (95% variance),517.565986,1093.310471,355.465719,0.317107
0,Baseline (Median),593.133125,1376.074250,340.397248,-0.093539


In [13]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

final_model = ridge_pca
final_model.fit(X_train, y_train)

pred = final_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("RMSE:", root_mean_squared_error(y_test, pred))
print("R²:", r2_score(y_test, pred))

MAE: 463.01201119381864
RMSE: 740.8525997322415
R²: 0.5125684219804726


In [15]:
comparison.to_csv(
    "/content/drive/MyDrive/Nhan_dang_mau/model_comparison.csv",
    index=False
)

In [16]:
report_text = f"""
BÀI TẬP: Dự báo giá xe ô tô bằng hồi quy và PCA

Dữ liệu: Bonbanh, 2.000 tin xe duy nhất.
Target: price_million_vnd (triệu VNĐ).

Kích thước dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột.

Kết quả Cross Validation:
{comparison.to_string(index=False)}

Kết quả trên tập kiểm tra:
MAE: {mean_absolute_error(y_test, pred):.2f}
RMSE: {root_mean_squared_error(y_test, pred):.2f}
R²: {r2_score(y_test, pred):.4f}
"""

with open(
    "/content/drive/MyDrive/Nhan_dang_mau/ket_qua_thuc_nghiem.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(report_text)